# Meme Sorting - Model Comparison
situation만 사용하여 두 임베딩 모델 비교
- `dragonkue/BGE-m3-ko` (1024d, max 8192 tokens)
- `jhgan/ko-sroberta-multitask` (768d, max 128 tokens)

In [1]:
print(1)

1


In [2]:
import os
import psycopg2
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# 두 모델 로드
model_bge = SentenceTransformer("dragonkue/BGE-m3-ko")
print("모델 로드 완료: dragonkue/BGE-m3-ko")

model_sroberta = SentenceTransformer("jhgan/ko-sroberta-multitask")
print("모델 로드 완료: jhgan/ko-sroberta-multitask")

모델 로드 완료: dragonkue/BGE-m3-ko
모델 로드 완료: jhgan/ko-sroberta-multitask


In [3]:
# DB에서 meme_examples 가져오기
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT", 5432),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

cur = conn.cursor()
cur.execute("""
    SELECT e.meme_id, e.example_id, e.situation, m.meme_name
    FROM meme_examples e
    JOIN memes m ON e.meme_id = m.meme_id
    ORDER BY e.meme_id, e.example_id
""")
rows = cur.fetchall()
cur.close()
conn.close()

print(f"총 {len(rows)}개 example 로드")

총 303개 example 로드


In [4]:
# 제품 설명 5개
item_descriptions = [
    "초경량 무선 블루투스 이어폰, 노이즈캔슬링 기능 탑재, 운동할 때 안 빠지는 이어폰",
    "유기농 수제 강아지 간식, 연어와 고구마로 만든 건강한 펫 트릿",
    "레트로 감성 필름 카메라, 감성 사진 찍기 좋은 빈티지 디자인",
    "1인용 캠핑 의자, 초경량 접이식, 배낭에 넣을 수 있는 사이즈",
    "대용량 보조배터리 20000mAh, 급속충전 지원, 여행 필수템",
]

situations = [r[2] for r in rows]

# 각 모델별 임베딩 생성
item_embs_bge = model_bge.encode(item_descriptions, normalize_embeddings=True)
sit_embs_bge = model_bge.encode(situations, normalize_embeddings=True)
print(f"BGE-m3-ko 임베딩 완료: 차원={item_embs_bge.shape[1]}")

item_embs_sro = model_sroberta.encode(item_descriptions, normalize_embeddings=True)
sit_embs_sro = model_sroberta.encode(situations, normalize_embeddings=True)
print(f"ko-sroberta 임베딩 완료: 차원={item_embs_sro.shape[1]}")

BGE-m3-ko 임베딩 완료: 차원=1024
ko-sroberta 임베딩 완료: 차원=768


In [10]:
TOP_N = 10

for idx, item_desc in enumerate(item_descriptions):
    top_bge = get_top_memes(item_embs_bge[idx:idx+1], sit_embs_bge, rows, TOP_N)
    top_sro = get_top_memes(item_embs_sro[idx:idx+1], sit_embs_sro, rows, TOP_N)

    print(f"{'='*80}")
    print(f"제품 {idx+1}: {item_desc}")
    print(f"{'='*80}")

    print(f"\n  [BGE-m3-ko]")
    for rank, m in enumerate(top_bge, 1):
        print(f"    {rank:>2}. [{m['similarity']:.4f}] {m['situation']}")

    print(f"\n  [ko-sroberta-multitask]")
    for rank, m in enumerate(top_sro, 1):
        print(f"    {rank:>2}. [{m['similarity']:.4f}] {m['situation']}")
    print()

제품 1: 초경량 무선 블루투스 이어폰, 노이즈캔슬링 기능 탑재, 운동할 때 안 빠지는 이어폰

  [BGE-m3-ko]
     1. [0.3519] 운동선수들이 운동하는 모습을 보여주는 영상
     2. [0.3138] 제품 홍보 캠페인
     3. [0.3077] 유통업계 댄스 마케팅 트렌드
     4. [0.3057] 친구와의 소통을 한층 더 자유롭게!
     5. [0.2996] 일상 속 노래 패러디
     6. [0.2838] 산책로를 걸으면서
     7. [0.2837] 패러디 쇼츠 콘텐츠
     8. [0.2834] 브랜드 협업 발표
     9. [0.2793] The video presents an animation meme
    10. [0.2789] 유튜브에서 한일커플의 소소한 일상 공유

  [ko-sroberta-multitask]
     1. [0.3321] 김동현 선수가 운동하면서 '운동 많이 된다'라고 말하는 장면
     2. [0.2756] 틱톡에서 발박수 챌린지를 촬영하는 중
     3. [0.2742] 무료 AI 케이팝 데몬 헌터스 필터 소개
     4. [0.2647] 노래 가사에서 반복적으로 '하지마'라는 가사가 나옴
     5. [0.2595] 노래방에서 노래를 부르는 영상
     6. [0.2483] 노래 'Golden'을 부르는 상황
     7. [0.2446] 노래 가사
     8. [0.2327] 게임이 망해가는 상황을 꽹과리 소리로 표현
     9. [0.2308] 유튜브 채널 '피식대학' 방송 중
    10. [0.2283] 안성재의 여러가지 슈트핏을 보여주는 영상

제품 2: 유기농 수제 강아지 간식, 연어와 고구마로 만든 건강한 펫 트릿

  [BGE-m3-ko]
     1. [0.4490] 유튜브 영상에서 강아지들이 함께 놀며 플랍하는 모습을 보여주는 콘텐츠
     2. [0.3535] 고양이가 점점 빨라지는 모습
     3. [0.3468] 오징어 게임에서 성기훈이

In [9]:
TOP_N = 10

for idx, item_desc in enumerate(item_descriptions):
    top_bge = get_top_memes(item_embs_bge[idx:idx+1], sit_embs_bge, rows, TOP_N)
    top_sro = get_top_memes(item_embs_sro[idx:idx+1], sit_embs_sro, rows, TOP_N)

    print(f"{'='*90}")
    print(f"제품 {idx+1}: {item_desc}")
    print(f"{'='*90}")
    print(f"{'':>4}  {'BGE-m3-ko':<55} | {'ko-sroberta-multitask'}")
    print(f"{'':>4}  {'-'*55} | {'-'*55}")
    for rank in range(TOP_N):
        b = top_bge[rank]
        s = top_sro[rank]
        b_text = f"[{b['similarity']:.4f}] {b['meme_name'][:20]}"
        s_text = f"[{s['similarity']:.4f}] {s['meme_name'][:20]}"
        print(f"    {rank+1:>2}. {b_text:<55} | {s_text}")
    print()

제품 1: 초경량 무선 블루투스 이어폰, 노이즈캔슬링 기능 탑재, 운동할 때 안 빠지는 이어폰
      BGE-m3-ko                                               | ko-sroberta-multitask
      ------------------------------------------------------- | -------------------------------------------------------
     1. [0.3519] 운동 많이 된다                                       | [0.3321] 운동 많이 된다
     2. [0.3138] 치킨 바나나                                         | [0.2756] 발박수 챌린지
     3. [0.3077] 랫 댄스                                           | [0.2742] 케이팝 데몬 헌터스
     4. [0.3057] 오우~ 즉시 반말?                                     | [0.2647] 하지마
     5. [0.2996] 트로트 ver.                                       | [0.2595] 매끈매끈하다 매끈매끈한
     6. [0.2838] 매끈매끈하다 매끈매끈한                                   | [0.2483] Golden
     7. [0.2837] 폭싹 속았수다                                        | [0.2446] Soda Pop
     8. [0.2834] 임짱                                             | [0.2327] 꽹과리론
     9. [0.2793] Sugar On My Tongue                             | [0.2308] 